# Lesson 01 - AIエージェント入門

**初心者向けAIエージェント**コースの最初のレッスンへようこそ！

**AIエージェント**とは、大規模言語モデル（LLM）を推論エンジンとして使用し、APIの呼び出し、データベースのクエリ、コードの実行など、ユーザーに代わって目標を達成するために現実世界で*アクション*を取ることができるプログラムのことです。

このノートブックでは、最初のエージェントである**旅行代理店エージェント**を構築します。これは休暇先をおすすめするものです。途中で以下のことを学びます:

1. **Microsoft Agent Framework**を使ってAzure AI Foundry Agent Serviceに接続する方法。
2. エージェントに**ツール**（呼び出すことができる単純なPython関数）を与える方法。
3. エージェントを実行し、その応答を検査する方法。
4. エージェントの応答をトークン単位でストリーム表示する方法。


## セットアップ

このノートブックを実行する前に、以下を確認してください：

1. **チャットモデルがデプロイされた Azure AI Foundry プロジェクト**（例: `gpt-4o-mini`）。
2. **Azure CLI にログイン済み** — ターミナルで `az login` を実行してください。
3. **必要な環境変数を設定済み:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry プロジェクトのエンドポイント。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — デプロイしたモデルの名前。

以下のセルでは必要なPythonパッケージをインストールします。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 最初のエージェントを作成する

エージェントには2つのものが必要です：

- **指示** — それが*誰であるか*と*どのように振る舞うか*を伝えるもの（システムプロンプト）。
- **ツール** — エージェントが情報を取得したりアクションを実行したりするために呼び出せる、`@tool`で装飾されたPython関数。

以下では、人気のある旅行先のリストを返すシンプルなツールを定義します。エージェントはユーザーが旅行のおすすめを求めたときにこのツールを使用します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ストリーミング応答

よりインタラクティブな体験のために、エージェントの応答を**ストリーミング**することができます。完全な返信を待つのではなく、エージェントは生成されるテキストの断片を逐次出力します。これは、チャットインターフェイスでリアルタイムに出力を表示したい場合に特に有用です。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

このレッスンでは以下を学びました：

- `AzureAIProjectAgentProvider` を介して Azure AI Foundry Agent Service に接続する **プロバイダーの作成**。
- エージェントが Python 関数を呼び出せるように **`@tool` デコレーターを使ったツールの定義**。
- ユーザーメッセージで **エージェントの実行とその応答の表示**。
- **リアルタイム出力のための応答のストリーミング**。

次のレッスンではエージェントフレームワークをより深く探り、エージェントにより強力なツールと多段階推論機能を与える方法を学びます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されています。正確性を期しておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご了承ください。原文はその言語における正式な資料としてご参照ください。重要な情報については、専門の人間翻訳者による翻訳を推奨いたします。本翻訳の利用により生じたいかなる誤解や解釈の違いについても、一切の責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
